# Conveyor Perception v2 — Coach-Powered Walkthrough

**The complete industrial CV stack on Colab T4, with a Gemini-powered Coach that diagnoses failures and reviews the run.**

This is the production demo of [roniejosephv-star/conveyor-perception](https://github.com/roniejosephv-star/conveyor-perception). It runs end-to-end on a free Colab T4 GPU and shows every part of the stack that maps to a real recycling-line JD:

- **Detection** (YOLO26 + OpenCV DNN, NMS-free, segmentation-aware fallback via UltralyticsDetector)
- **Tracking** (supervision ByteTrack, IoU fallback for tests)
- **Drift detection** (3-signal: KS test on confidence, z-score on counts, MAD on latency)
- **L1 triage** (7 deterministic severity rules + MCP-style 5-tool surface)
- **Predictive maintenance** (rule-based drift signals → actionable hints)
- **Robustness** (13 MRF-condition augmentations, broken/degraded/ok classification)
- **Monitoring** (FastAPI-style shift dashboard + retrain recommendation)

**Runtime**: ~20 min on free T4 (12 min training + 8 min walkthrough).

**The Coach**: an optional Gemini integration that reads the session log and diagnoses any failures. Set the `GEMINI_API_KEY` Colab secret (key icon in the left sidebar) to enable it. Without a key, the Coach still works — it falls back to static hints.

---

## How to use this notebook

1. Runtime → Change runtime type → **T4 GPU** (already done if you see the green check)
2. Click **Run all** in the Runtime menu, OR walk through cells one at a time
3. Each cell logs to a shared `state` (SessionState singleton). Errors are caught and stored.
4. The Coach cell (§4 cell 15) asks Gemini to diagnose any failures automatically.
5. The final cell (§4 cell 16) offers a JSON download of the full session log.

**Toggle modules** in §1 cell 4 to enable/disable each of the 4 abstractions and 8 modules. The pipeline reads these toggles and skips disabled components.


In [ ]:
# --- Cell 1: Runtime + env check ---
REPO = '/content/conveyor-perception'
REPO_URL = 'https://github.com/roniejosephv-star/conveyor-perception.git'
import os, sys
os.chdir(REPO) if os.path.exists(REPO) else None  # if not cloned yet, this no-ops
sys.path.insert(0, REPO)
sys.path.insert(0, os.path.join(REPO, 'notebooks'))

# --- Self-heal: clone the repo if colab_session is not importable ---
# Cell 1 is the first cell the user typically runs, and it needs colab_session
# from the cloned repo. If the user hasn't run cell 2 yet, the repo doesn't
# exist and the import fails. We clone here so the notebook works in any
# order. Idempotent and fast (~3-5s on first run, instant after).
try:
    from colab_session import env_check, get_state  # noqa: F401
except ImportError:
    import subprocess
    print('Repo not found at /content/conveyor-perception — cloning (~5s)...')
    subprocess.run(['git', 'clone', REPO_URL, REPO], check=True)
    sys.path.insert(0, REPO)
    sys.path.insert(0, os.path.join(REPO, 'notebooks'))
    os.chdir(REPO)
    from colab_session import env_check, get_state  # noqa: F401 (after clone)

# Load the session helpers
from colab_session import env_check, get_state

state = get_state()
state.env = env_check()

print('=' * 60)
print(f"  GPU:       {state.env.get('gpu', 'unknown')}")
print(f"  RAM:       {state.env.get('ram_gb', '?')} GB")
print(f"  Disk free: {state.env.get('disk_gb_free', '?')} GB")
print(f"  Python:    {state.env.get('python', '?')}")
print(f"  In Colab:  {state.env.get('is_colab', False)}")
print('=' * 60)

# Soft checks — warn but don't fail
if state.env.get('gpu') == 'CPU':
    print('\n⚠ Running on CPU. The pipeline still works but inference will be ~10x slower.'
          ' Switch to T4 GPU in Runtime → Change runtime type.')
if state.env.get('ram_gb', 0) < 10:
    print(f"\n⚠ Only {state.env.get('ram_gb', '?')} GB RAM. Some cells may need --batch 16 instead of 32.")
if state.env.get('disk_gb_free', 0) < 5:
    print(f"\n⚠ Only {state.env.get('disk_gb_free', '?')} GB free disk. Dataset + model need ~2 GB.")

state.log('cell-1', action='env-check', env=state.env)
print('\n✓ Cell 1 done. State initialized.')


---

## §1 SETUP — runtime check, install, state, toggles

Get a clean T4 environment, install pinned deps, clone the repo, set up the shared state, and pick which modules to run.


In [ ]:
# --- Cell 2: Install + clone + Roboflow key ---
import os, subprocess, sys
from pathlib import Path

REPO = Path('/content/conveyor-perception')

with state.cell('cell-2', action='install-and-clone'):
    # Install pinned deps (matches requirements.txt)
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q',
        'ultralytics==8.4.121',
        'opencv-python==4.11.0.86',
        'supervision==0.30.0',
        'fastmcp==3.4.7',
        'pydantic==2.13.4',
        'roboflow==1.4.1',
        'onnxruntime>=1.20.1',
        'numpy>=1.26,<2.0',
        'pyyaml==6.0.2',
        'python-dotenv>=1.1.0',
        'ipywidgets>=8.0',
        'google-generativeai>=0.8',
    ], check=True)
    print('✓ Pinned deps installed')

    # Clone or pull the repo
    if not REPO.exists():
        subprocess.run([
            'git', 'clone',
            'https://github.com/roniejosephv-star/conveyor-perception.git',
            str(REPO),
        ], check=True)
        print(f'✓ Cloned repo to {REPO}')
    else:
        subprocess.run(['git', '-C', str(REPO), 'pull', '--rebase'], check=False)
        print(f'✓ Pulled latest from {REPO}')

    # Roboflow API key (public read-only key for the demo; replace for real use)
    if not os.path.exists(REPO / '.env'):
        with open(REPO / '.env', 'w') as f:
            f.write('ROBOFLOW_API_KEY=qogO5hAuLgUUYMbNT6W3\n')
        print('✓ Wrote demo .env (read-only public key; replace for real work)')
    else:
        print('✓ .env already present')

    # Add to path
    sys.path.insert(0, str(REPO))
    sys.path.insert(0, str(REPO / 'notebooks'))
    os.chdir(REPO)

print('\n✓ Cell 2 done. Repo ready.')


In [ ]:
# --- Cell 3: Initialize SessionState + reload env ---
from colab_session import get_state, reset_state, env_check

state = get_state()
state.env = env_check()  # re-check now that we have the right env
state.metric('session_started', state.session_id)

print(f'Session: {state.session_id}')
print(f'Env: {state.env["gpu"]} · {state.env["ram_gb"]} GB RAM')
print(f'Toggles: {sum(state.toggles.values())}/{len(state.toggles)} enabled (all on by default)')
print('\nEvery subsequent cell will log to `state`. Errors are caught and stored.')
print('\n✓ Cell 3 done. State ready.')


In [ ]:
# --- Cell 4: Module toggle UI ---
# Tick / untick to enable / disable each component. The pipeline cells
# in §2 read state.toggles to decide what to instantiate.

from colab_session import toggle_ui, get_state

ui = toggle_ui()
display(ui)

state = get_state()
print('\nCurrent toggles:')
for k, v in state.toggles.items():
    icon = '✓' if v else '○'
    print(f'  {icon} {k}')


---

## §2 WALKTHROUGH — the 4 abstractions + 7 modules

Each component is loaded in its own cell so failures are isolated. If any cell errors, the Coach (§4) will diagnose it.


In [ ]:
# --- Cell 6: The 4 framework abstractions ---
import sys, os
sys.path.insert(0, '/content/conveyor-perception')
os.chdir('/content/conveyor-perception')

from colab_session import get_state, hint_for
from conveyor_perception.core.detection_pipeline import Detector, Detection
from conveyor_perception.core.tracking_pipeline import TrackingPipeline
from conveyor_perception.core.drift_monitor import DriftMonitor
from conveyor_perception.core.triage_surface import MCPTriageSurface

state = get_state()
loaded = {}

with state.cell('cell-6', action='load-4-abstractions'):
    if state.toggles.get('abstraction:detector'):
        # Detector loads ONNX; we'll wire the model in cell 8 after training.
        # For now just verify the class imports.
        loaded['detector_class'] = Detector
        print('✓ Detector class loaded (YOLO26 + OpenCV DNN)')

    if state.toggles.get('abstraction:tracker'):
        loaded['tracker'] = TrackingPipeline()
        print('✓ TrackingPipeline instantiated (ByteTrack with IoU fallback)')

    if state.toggles.get('abstraction:drift_monitor'):
        loaded['drift_monitor'] = DriftMonitor(baseline_window=50, min_samples_for_drift=20)
        print('✓ DriftMonitor instantiated (KS test + z-score + MAD)')

    if state.toggles.get('abstraction:triage'):
        loaded['triage_surface'] = MCPTriageSurface()
        print('✓ MCPTriageSurface instantiated (5 tools, FastMCP server)')

print(f'\nLoaded: {len(loaded)}/4 abstractions')
state.log('cell-6', action='result', loaded=list(loaded.keys()))


In [ ]:
# --- Cell 7: The 7+1 JD modules — show signatures and import paths ---
import importlib, inspect
from colab_session import get_state

state = get_state()

modules_meta = [
    ('module:perception',           'conveyor_perception.perception',   'Detector + UltralyticsDetector'),
    ('module:triage',               'conveyor_perception.triage',       'L1TriageAgent + 7 severity rules'),
    ('module:predictive_maintenance', 'conveyor_perception.predictive_maintenance', 'MaintenanceAdvisor + 3 signal types'),
    ('module:multitask',            'conveyor_perception.multitask',    'MultitaskPipeline (Detector→Tracker→Drift→Triage)'),
    ('module:integration',          'conveyor_perception.integration',  'ConveyorNode (real ROS 2) + MockROS2Node (CI)'),
    ('module:robustness',           'conveyor_perception.robustness',   'RobustnessTestSuite + 13 augmentations'),
    ('module:monitoring',           'conveyor_perception.monitoring',   'MonitoringDashboard + ShiftReport'),
    ('module:optimization',         'conveyor_perception.optimization', 'benchmark_pytorch/onnx + export_onnx'),
]

loaded = []
skipped = []
for toggle_key, module_path, desc in modules_meta:
    if not state.toggles.get(toggle_key):
        skipped.append(toggle_key)
        print(f'  ○ {module_path} (disabled by toggle)')
        continue
    try:
        with state.cell(f'cell-7-{module_path}', action='import'):
            importlib.import_module(module_path)
            loaded.append(module_path)
            print(f'  ✓ {module_path} — {desc}')
    except Exception as exc:
        print(f'  ✗ {module_path} failed: {exc}')
        print(f'    Hint: {hint_for(exc)}')

print(f'\nLoaded: {len(loaded)}/{len(modules_meta)} modules, skipped: {len(skipped)}')
state.metric('modules_loaded', len(loaded))
state.metric('modules_skipped', len(skipped))


In [ ]:
# --- Cell 8: Train YOLO26s (or skip to pretrained for the fast path) ---
import os, sys, subprocess, time
from pathlib import Path
from colab_session import get_state, hint_for

state = get_state()
REPO = Path('/content/conveyor-perception')

# Choose: full training (12 min) or pretrained (auto-download)
TRAIN_MODE = 'train'  # 'train' or 'pretrained'

if TRAIN_MODE == 'train':
    print('Training YOLO26s on T4 (30 epochs, ~10-15 min)...')
    print('Set TRAIN_MODE = "pretrained" above to skip training and use COCO weights.\n')
    t0 = time.time()
    with state.cell('cell-8', action='train-yolo26s'):
        # Download dataset first
        result = subprocess.run([
            sys.executable, 'scripts/download_dataset.py',
        ], capture_output=True, text=True, cwd=REPO)
        if result.returncode != 0:
            print('Dataset download failed:')
            print(result.stderr[-500:])
            print(f'Hint: {hint_for(Exception(result.stderr))}')
        else:
            print(result.stdout[-500:])

        # Train (will pick up the model from last.pt if it exists)
        result = subprocess.run([
            sys.executable, 'scripts/train_yolo26.py',
            '--epochs', '30',
            '--imgsz', '640',
            '--batch', '16',  # conservative for free T4
            '--device', '0',
        ], capture_output=True, text=True, cwd=REPO)
        if result.returncode != 0:
            print('Training failed:')
            print(result.stderr[-500:])
        else:
            print(result.stdout[-500:])

        train_time = time.time() - t0
        state.metric('train_time_sec', round(train_time, 1))
        print(f'\n✓ Training complete in {train_time/60:.1f} min')
else:
    print('Using COCO pretrained YOLO26s (fast path, ~30s download)')
    with state.cell('cell-8', action='download-pretrained'):
        from ultralytics import YOLO
        YOLO('yolo26s.pt')  # auto-downloads
        print('✓ Pretrained weights ready')


In [ ]:
# --- Cell 9: End-to-end pipeline (Detector→Tracker→Drift→Triage→Maintenance) ---
import sys, os, time, urllib.request
import numpy as np
sys.path.insert(0, '/content/conveyor-perception')
os.chdir('/content/conveyor-perception')

from colab_session import get_state, hint_for
from conveyor_perception.core.drift_monitor import DriftMonitor
from conveyor_perception.core.tracking_pipeline import TrackingPipeline
from conveyor_perception.multitask.pipeline import MultitaskPipeline
from conveyor_perception.perception.detector import Detector
from conveyor_perception.perception.ultralytics_detector import UltralyticsDetector
from conveyor_perception.predictive_maintenance.advisor import DriftSignal, MaintenanceAdvisor
from conveyor_perception.triage.agent import L1TriageAgent
from conveyor_perception.monitoring.dashboard import MonitoringDashboard

state = get_state()

with state.cell('cell-9', action='run-pipeline'):
    # Use the UltralyticsDetector (handles both .pt and seg-trained .onnx)
    model_path = 'yolo26s_recyclable.pt' if os.path.exists('models/yolo26s_recyclable.pt') else 'yolo26s.pt'
    class_names = ['Glass', 'metal', 'plastic', 'vinyl'] if 'recyclable' in model_path else \
        [f'class_{i}' for i in range(80)]  # COCO fallback

    det = UltralyticsDetector(
        model_path=model_path,
        class_names=class_names,
        conf_threshold=0.25,
        device='cuda:0',
        imgsz=640,
    )
    tracker = TrackingPipeline()
    drift = DriftMonitor(baseline_window=50, min_samples_for_drift=20)
    triage = L1TriageAgent()
    advisor = MaintenanceAdvisor()
    dashboard = MonitoringDashboard()
    pipeline = MultitaskPipeline(det, tracker, drift, triage)

    # Get a sample image (real recycling if downloaded, else COCO bus)
    sample_path = '/content/conveyor-perception/data/sample/bus.jpg'
    if not os.path.exists(sample_path):
        os.makedirs(os.path.dirname(sample_path), exist_ok=True)
        urllib.request.urlretrieve('https://ultralytics.com/images/bus.jpg', sample_path)
    import cv2
    image = cv2.imread(sample_path)
    print(f'Sample image: {image.shape}')

    # Run 30 frames to accumulate drift signals
    print('\nRunning 30 frames through the pipeline...')
    t0 = time.perf_counter()
    last_result = None
    for i in range(30):
        last_result = pipeline.step(image)
        dashboard.record_frame(last_result)
    elapsed = (time.perf_counter() - t0) * 1000
    inference_ms = elapsed / 30
    state.metric('t4_inference_ms', round(inference_ms, 2))
    print(f'\n✓ Pipeline ran 30 frames in {elapsed/1000:.1f}s ({inference_ms:.1f} ms/frame on T4)')
    print(f'  Last frame: {len(last_result.detections)} detections, {len(last_result.alerts)} alerts')


In [ ]:
# --- Cell 10: Triage queue, robustness suite, shift dashboard ---
import json, sys, os
sys.path.insert(0, '/content/conveyor-perception')
os.chdir('/content/conveyor-perception')

from colab_session import get_state
from conveyor_perception.robustness import RobustnessTestSuite

state = get_state()

# 1. Triage queue (most recent alerts)
print('=== Triage Queue (most recent 10 alerts) ===\n')
for alert in triage.get_pending(limit=10):
    print(f"  [{alert.severity.upper():9s}] {alert.class_name:10s} conf={alert.confidence:.2f} reason='{alert.metadata.get('reason', '')[:50]}'")

# 2. Robustness suite (13 MRF conditions)
print('\n=== Robustness Suite ===\n')
with state.cell('cell-10-robustness', action='run-robustness'):
    import cv2
    image = cv2.imread('/content/conveyor-perception/data/sample/bus.jpg')
    suite = RobustnessTestSuite(det, image)
    report = suite.run()
    print(report.to_markdown())
    state.metric('robustness_verdict', report.verdict)

# 3. Shift dashboard
print('\n=== Shift Dashboard ===\n')
shift = dashboard.shift_report()
print(json.dumps(shift.to_dict(), indent=2))
state.metric('retrain_recommended', shift.retrain_recommended)


In [ ]:
# --- Cell 11: Coach review (preview) ---
# A quick Gemini review of the run so far. Full review in §4 cell 15.
from colab_session import get_state, coach_review

state = get_state()
print('Asking the Coach for a quick review...\n')
review = coach_review(state)
print(review)


---

## §3 COMPARISON — the prototype vs EverestLabs' stack

The same code, on the same class of GPU that EverestLabs ships. The numbers below come from this T4 run plus EverestLabs' published spec.


In [ ]:
# --- Cell 12: T4 vs EverestLabs (the comparison) ---
from colab_session import get_state

state = get_state()

# Published numbers from EverestLabs' spec (the target we benchmark against)
EVEREST_PUBLISHED = {
    'gpu': 'RTX 2000 Ada (Innodisk APEX-P200)',
    'classes': 60,
    'classification_ms': '8-12',
    'fps': 30,
    'accuracy_pct': 95,
    'pick_success_pct': 90,
}

# T4 measured numbers come from cell 9
t4_inference = state.metrics.get('t4_inference_ms', 'not measured yet')
T4_MEASURED = {
    'gpu': 'Colab T4 (similar class to RTX 2000 Ada)',
    'classes': 4,
    'inference_ms': t4_inference,
    'fps': round(1000 / t4_inference, 1) if isinstance(t4_inference, (int, float)) else 'n/a',
    'training_minutes': 12,  # 30 epochs on T4
    'mAP50': 'TBD (depends on full training)',
}

import pandas as pd
df = pd.DataFrame([
    {'Metric': 'GPU', 'EverestLabs': EVEREST_PUBLISHED['gpu'], 'T4 (this run)': T4_MEASURED['gpu']},
    {'Metric': 'Classes', 'EverestLabs': EVEREST_PUBLISHED['classes'], 'T4 (this run)': T4_MEASURED['classes']},
    {'Metric': 'Inference (ms)', 'EverestLabs': EVEREST_PUBLISHED['classification_ms'], 'T4 (this run)': T4_MEASURED['inference_ms']},
    {'Metric': 'FPS', 'EverestLabs': EVEREST_PUBLISHED['fps'], 'T4 (this run)': T4_MEASURED['fps']},
    {'Metric': 'mAP@50', 'EverestLabs': '95% accuracy', 'T4 (this run)': T4_MEASURED['mAP50']},
    {'Metric': 'Pick success', 'EverestLabs': f"{EVEREST_PUBLISHED['pick_success_pct']}%", 'T4 (this run)': 'n/a (no robot)'},
])

print('=== Hardware Stack Comparison ===\n')
print(df.to_string(index=False))
print()
print('Reading the table:')
print('  • The T4 is the same class as the RTX 2000 Ada (Turing/Ampere gen, similar INT8 TOPS).')
print('  • Our 4-class model is a prototype — Everest has 60+ in production.')
print('  • The mAP50 is for our 4-class recycling subset. Everest publishes 95% accuracy on 60 classes.')

state.log('cell-12', action='comparison', t4_inference_ms=t4_inference)


---

## §4 COACH — error log, diagnosis, summary, publish

The Coach reads `state.errors` and asks Gemini to diagnose each one. Without a Gemini key, the Coach falls back to static hints (still useful).

Set `GEMINI_API_KEY` in the Colab secrets panel (key icon, left sidebar) to enable AI diagnosis. Set `GITHUB_TOKEN` (classic PAT, scope: repo) to enable the optimization loop — the final cell publishes the run as a GitHub Release and a GitHub Action picks it up to suggest code improvements as a PR.

Without the tokens, the notebook still works: errors get diagnosed (via static hints), and the session log gets downloaded (via the browser).


In [ ]:
# --- Cell 13: Error log + Coach diagnosis ---
import json
from colab_session import get_state, coach_diagnose, hint_for

state = get_state()

if not state.has_errors():
    print('✓ No errors captured. The pipeline ran clean.')
else:
    print(f'\n{len(state.errors)} error(s) captured during this run.\n')
    print('=' * 60)

    for i, err in enumerate(state.errors, 1):
        print(f'\n### Error {i}/{len(state.errors)} — {err["cell_id"]}')
        print(f'  Type:    {err["type"]}')
        print(f'  Message: {err["message"][:200]}')
        if err.get('hint'):
            print(f'  Static hint: {err["hint"]}')
        print()

        # Ask the Coach to diagnose
        extra = f"Session: {state.session_id}. Env: {state.env.get('gpu', '?')}."
        with state.cell(f'cell-13-diagnose-{i}', action='coach-diagnose'):
            diagnosis = coach_diagnose(err, extra_context=extra)
            print(f'**Coach diagnosis:**\n\n{diagnosis}\n')
            state.gemini_diagnoses.append({
                'error_idx': i,
                'cell_id': err['cell_id'],
                'diagnosis': diagnosis,
            })
        print('-' * 60)


In [ ]:
# --- Cell 14: Summary + downloadable session log ---
import json
from colab_session import get_state, summary_table, download_session_log, coach_review

state = get_state()

print('### Session Summary\n')
print(state.summary_table())

print('\n### Full Coach Review (post-run)\n')
review = coach_review(state)
print(review)

# Offer the download
print('\n### Download session log\n')
try:
    download_session_log()
    print('(Browser download triggered. If nothing happened, check your browser popup blocker.)')
except Exception as e:
    print(f'Download failed: {e}')
    print('You can still access the log via: state.to_json()')

print('\n✓ Cell 14 done. Session complete.')
print(f'\nFinal state: {len(state.logs)} log entries, {len(state.errors)} errors, {len(state.metrics)} metrics.')
print(f'Gemini diagnoses: {len(state.gemini_diagnoses)}')


In [ ]:
# --- Cell 15: Publish to GitHub Release (kicks off the optimization loop) ---
# This cell uploads the session log as a GitHub Release asset. The release
# tag is v0.0.{N} where N = number of existing releases + 1. A GitHub
# Action triggers on release-published, downloads the log, asks Gemini
# to suggest improvements, and opens a PR.

import os, json
from colab_session import get_state

# --- Self-healing: ensure PyGithub is installed (idempotent, fast if cached) ---
# Colab doesn't ship PyGithub by default, and the install cell can be
# masked by pip dependency-resolver warnings, so we re-check here and install
# if needed. Uses --no-deps to avoid the numpy 1.26 / 2.x conflict.
try:
    from github import Github  # noqa: F401  (PyGithub)
except ImportError:
    import subprocess, sys
    print('PyGithub not found — installing (one-time, ~5s)...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--no-deps', 'PyGithub', '-q'])
    from github import Github  # noqa: F401  (PyGithub, after install)

state = get_state()

REPO = 'roniejosephv-star/conveyor-perception'

# GitHub PAT (read+write to your repo). Get one at
# https://github.com/settings/tokens (classic PAT, scope: repo).
try:
    from google.colab import userdata  # type: ignore
    gh_token = userdata.get('GITHUB_TOKEN')
except Exception:
    gh_token = os.environ.get('GITHUB_TOKEN')

if not gh_token:
    print('=' * 60)
    print('  No GITHUB_TOKEN configured.\n')
    print('  To enable the optimization loop:')
    print('  1. Create a PAT at https://github.com/settings/tokens')
    print('     (Classic, scope: repo, expiry: 90 days)')
    print('  2. In Colab, click the key icon and add:')
    print('     Name: GITHUB_TOKEN')
    print('     Value: <paste the PAT>')
    print('     Toggle notebook access: ON')
    print('  3. Re-run this cell.')
    print('=' * 60)
    print('\n✓ Cell 15 done (publish skipped).')
    state.log('cell-15', action='publish-skipped', reason='no GITHUB_TOKEN')
else:
    from github import Github  # PyGithub
    g = Github(gh_token)
    repo = g.get_repo(REPO)
    # Find the next version. v0.0.{N} where N = max existing + 1
    existing = list(repo.get_releases())
    next_n = 0
    for r in existing:
        tag = r.tag_name
        if tag.startswith('v0.0.'):
            try:
                n = int(tag.split('.')[-1])
                next_n = max(next_n, n + 1)
            except ValueError:
                pass
    new_tag = f'v0.0.{next_n}'

    # Write the session log to a temp file
    log_path = f'/tmp/{state.session_id}.json'
    with open(log_path, 'w') as f:
        f.write(state.to_json())

    # Build the release notes (one-liner with the headline metric)
    headline = state.metrics.get('t4_inference_ms', 'n/a')
    n_errors = len(state.errors)
    n_modules_on = sum(state.toggles.values())
    notes = (
        f'## Run {new_tag}\n\n'
        f'- **T4 inference (ms)**: {headline}\n'
        f'- **Errors**: {n_errors}\n'
        f'- **Modules on**: {n_modules_on}/{len(state.toggles)}\n'
        f'- **Session ID**: {state.session_id}\n\n'
        '_Auto-published by the Conveyor Perception Coach from the Colab demo._'
    )

    with state.cell('cell-15', action='publish-release'):
        release = repo.create_git_release(
            tag=new_tag,
            name=f'Run {new_tag} — T4 {headline}ms',
            message=notes,
            draft=False,
            prerelease=False,
        )
        # Attach the session log as a release asset
        release.upload_asset_from_path(log_path, name='session.json')
        state.metric('release_tag', new_tag)
        state.metric('release_url', release.html_url)
        print(f'\n✓ Published {new_tag} → {release.html_url}')
        print('  Asset: session.json')
        print('  The optimization loop will pick this up on the next Action run.')

    print(f'\n✓ Cell 15 done. Session published as {new_tag}.')
